# Beam Noise Diffusion (BND) Experiment

This notebook clones the latest code from GitHub into the Colab runtime. Google Drive is used only for persistent runtime artifacts: Hugging Face cache, generated images, result JSON files, and CSV summaries.


## Step 0: Check GPU

Verify which GPU Colab assigned before loading the diffusion model.


In [ ]:
!nvidia-smi

Sat May 23 18:58:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Mount Google Drive

Drive is used only as persistent storage for `cache/` and `outputs/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Clone Code from GitHub

The repository is cloned into the temporary Colab runtime. This keeps executable code separate from persistent outputs.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/nguyenphamngocquy/beam-noise-diffusion.git"
CODE_ROOT = Path("/content/beam-noise-diffusion")
STORAGE_ROOT = Path("/content/drive/MyDrive/beam-noise-diffusion-storage")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT))

print("Code root:", CODE_ROOT)
print("Storage root:", STORAGE_ROOT)

Code root: /content/beam-noise-diffusion
Storage root: /content/drive/MyDrive/beam-noise-diffusion-storage


## Step 3: Install Dependencies

Install dependencies from the cloned repository's `requirements.txt`.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(CODE_ROOT / "requirements.txt"),
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/beam-noise-diffusion/requirements.txt'], returncode=0)

## Step 4: Load Config and Prompts

Configs and prompts are read from the cloned repository. Runtime outputs are written to `STORAGE_ROOT` on Drive.


In [ ]:
import os
from bnd.config import load_config, load_prompts
from bnd.io import ensure_experiment_dirs
from google.colab import userdata

CONFIG_NAME = "bnd_full"  # Change to "bnd_full" for A100/full run
ensure_experiment_dirs(STORAGE_ROOT)
CFG = load_config(CODE_ROOT, CONFIG_NAME)
PROMPTS = load_prompts(CODE_ROOT)
RUN_NAME = CFG["name"]

hf_token = userdata.get("HF_TOKEN")
if hf_token is None:
    raise RuntimeError("Missing HF_TOKEN in Colab Secrets")

os.environ["HF_TOKEN"] = hf_token

print("Config:", RUN_NAME)
print("Perturbation:", CFG.get("perturbation_mode", "additive"))
print("Number of prompts:", len(PROMPTS))
print(PROMPTS[0])

Config: bnd_light
Perturbation: prior_mix
Number of prompts: 30
{'id': '0001', 'group': 'simple', 'prompt': 'a photo of a golden retriever sitting in a park'}


## Step 5: Load Models

Stable Diffusion and CLIP are loaded from code in the cloned repo. Model files are cached on Drive under `STORAGE_ROOT/cache/hf_cache`.


In [6]:
from bnd.sampling import get_gpu_name, load_sd_pipeline
from bnd.scoring import ClipScorer

print("GPU:", get_gpu_name())
CACHE_DIR = STORAGE_ROOT / "cache" / "hf_cache"
pipe = load_sd_pipeline(CFG, CACHE_DIR)
scorer = ClipScorer(CACHE_DIR)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /content/drive/MyDrive/beam-noise-diffusion-storage/cache/hf_cache/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


xFormers enabled.


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## Step 6: Run Experiments

For each prompt, the notebook runs three methods: random baseline, Monte-Carlo selection, and BND. Set `PROMPT_LIMIT` to a small number for a smoke test.


In [7]:
from bnd.algorithm import run_bnd, run_monte_carlo, run_random_baseline

PROMPT_LIMIT = None  # Set to a small number, e.g. 2, for a quick smoke test
items = PROMPTS if PROMPT_LIMIT is None else PROMPTS[:PROMPT_LIMIT]

for item in items:
    run_random_baseline(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_monte_carlo(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_bnd(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)

  0%|          | 0/25 [00:00<?, ?it/s]

[DONE] random 0001 score=0.3342


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

[DONE] bnd 0030 score=0.3506


## Step 7: Summarize Results

Collect JSON result files from Drive into CSV summaries and display aggregate metrics.


In [8]:
from bnd.io import summarize_results

df, summary, group_summary = summarize_results(STORAGE_ROOT, RUN_NAME)
display(df.head())
display(summary)
display(group_summary)

,prompt_id,prompt,group,method,score,runtime_sec,gpu,image_path
0,0001,a photo of a golden retriever sitting in a park,simple,random,0.334239,7.489705,Tesla T4,/content/drive/MyDrive/beam-noise-diffusion-st...
1,0002,a red apple on a wooden table,simple,random,0.293889,4.140688,Tesla T4,/content/drive/MyDrive/beam-noise-diffusion-st...
2,0003,a small boat floating on a calm lake,simple,random,0.311379,4.173616,Tesla T4,/content/drive/MyDrive/beam-noise-diffusion-st...
3,0004,a white cat sleeping on a sofa,simple,random,0.332284,4.192405,Tesla T4,/content/drive/MyDrive/beam-noise-diffusion-st...
4,0005,a mountain landscape during sunset,simple,random,0.308233,4.117079,Tesla T4,/content/drive/MyDrive/beam-noise-diffusion-st...


,method,mean_clipscore,std_clipscore,mean_runtime,count
0,bnd,0.335736,0.027635,49.404047,30
1,monte_carlo,0.333087,0.027877,28.475843,30
2,random,0.316826,0.026859,4.265438,30


,group,method,mean_clipscore,mean_runtime,count
0,attributes,bnd,0.351785,49.458947,10
1,attributes,monte_carlo,0.349424,28.506141,10
2,attributes,random,0.328313,4.152378,10
3,complex,bnd,0.332527,49.519745,10
4,complex,monte_carlo,0.329799,28.449539,10
5,complex,random,0.312111,4.157526,10
6,simple,bnd,0.322895,49.233449,10
7,simple,monte_carlo,0.320038,28.471849,10
8,simple,random,0.310054,4.486409,10


In [ ]:
from google.colab import runtime

# Lệnh ngắt kết nối và tắt hẳn Runtime Colab ngay lập tức
runtime.unassign()